In [31]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
!pip install fastapi uvicorn pyngrok nest_asyncio shap joblib --quiet

In [32]:
import pandas as pd
import numpy as np

# Model Loading
import joblib

# SHAP
import shap

# FastAPI
from fastapi import FastAPI
from pydantic import BaseModel

# Server
import uvicorn
from pyngrok import ngrok
import nest_asyncio
import threading

# Display
from IPython.display import display

print("All libraries imported successfully!")

All libraries imported successfully!


In [33]:
PROJECT_PATH = "/content/drive/MyDrive/AI-Relationship-Manager"

print(PROJECT_PATH)

/content/drive/MyDrive/AI-Relationship-Manager


In [34]:
import os

print("Project Folders:\n")

for folder in os.listdir(PROJECT_PATH):
    print("-", folder)

Project Folders:

- models
- notebooks
- outputs
- presentation
- data
- app


In [35]:
xgboost_model = joblib.load(
    f"{PROJECT_PATH}/models/xgboost_model.pkl"
)

print("XGBoost Model Loaded Successfully!")

XGBoost Model Loaded Successfully!


In [36]:
preprocessor = joblib.load(
    f"{PROJECT_PATH}/models/preprocessor.pkl"
)

print("Preprocessor Loaded Successfully!")

Preprocessor Loaded Successfully!


In [38]:
import os

print(os.listdir(f"{PROJECT_PATH}/models"))

['xgboost_model.pkl', 'preprocessor.pkl']


In [39]:
import shap

explainer = shap.TreeExplainer(xgboost_model)

print(type(explainer))
print("SHAP Explainer Created Successfully!")

<class 'shap.explainers._tree.TreeExplainer'>
SHAP Explainer Created Successfully!


In [40]:
import joblib

joblib.dump(
    explainer,
    f"{PROJECT_PATH}/models/shap_explainer.pkl"
)

print("SHAP Explainer Saved Successfully!")

SHAP Explainer Saved Successfully!


In [41]:
import os

print(os.listdir(f"{PROJECT_PATH}/models"))

['xgboost_model.pkl', 'preprocessor.pkl', 'shap_explainer.pkl']


In [42]:
feature_translator = {

    # Numeric Features
    "Tenure Months": "Customer has relatively short tenure.",
    "Monthly Charges": "Customer has high monthly charges.",
    "Total Charges": "Customer has high lifetime value.",

    # Gender
    "Gender_Female": "Customer is female.",
    "Gender_Male": "Customer is male.",

    # Senior Citizen
    "Senior Citizen_No": "Customer is not a senior citizen.",
    "Senior Citizen_Yes": "Customer is a senior citizen.",

    # Partner
    "Partner_No": "Customer does not have a partner.",
    "Partner_Yes": "Customer has a partner.",

    # Dependents
    "Dependents_No": "Customer has no dependents.",
    "Dependents_Yes": "Customer has dependents.",

    # Phone Service
    "Phone Service_No": "Customer does not use phone service.",
    "Phone Service_Yes": "Customer uses phone service.",

    # Multiple Lines
    "Multiple Lines_No": "Customer does not use multiple phone lines.",
    "Multiple Lines_Yes": "Customer uses multiple phone lines.",

    # Internet
    "Internet Service_DSL": "Customer uses DSL internet.",
    "Internet Service_Fiber optic": "Customer uses Fiber Optic internet.",
    "Internet Service_No": "Customer has no internet service.",

    # Online Security
    "Online Security_No": "Customer has not subscribed to Online Security.",
    "Online Security_Yes": "Customer has Online Security.",

    # Online Backup
    "Online Backup_No": "Customer has not subscribed to Online Backup.",
    "Online Backup_Yes": "Customer has Online Backup.",

    # Device Protection
    "Device Protection_No": "Customer has not subscribed to Device Protection.",
    "Device Protection_Yes": "Customer has Device Protection.",

    # Tech Support
    "Tech Support_No": "Customer has not subscribed to Tech Support.",
    "Tech Support_Yes": "Customer has Tech Support.",

    # Streaming TV
    "Streaming TV_No": "Customer has not subscribed to Streaming TV.",
    "Streaming TV_Yes": "Customer has Streaming TV.",

    # Streaming Movies
    "Streaming Movies_No": "Customer has not subscribed to Streaming Movies.",
    "Streaming Movies_Yes": "Customer has Streaming Movies.",

    # Contract
    "Contract_Month-to-month": "Customer is on a month-to-month contract.",
    "Contract_One year": "Customer has a one-year contract.",
    "Contract_Two year": "Customer has a two-year contract.",

    # Paperless Billing
    "Paperless Billing_No": "Customer does not use paperless billing.",
    "Paperless Billing_Yes": "Customer uses paperless billing.",

    # Payment
    "Payment Method_Bank transfer (automatic)": "Customer pays via bank transfer.",
    "Payment Method_Credit card (automatic)": "Customer pays via credit card.",
    "Payment Method_Electronic check": "Customer pays using electronic check.",
    "Payment Method_Mailed check": "Customer pays using mailed check."
}

print("Feature Translator Loaded!")
print("Total Translations:", len(feature_translator))

Feature Translator Loaded!
Total Translations: 39


In [44]:
for key in feature_translator:
    print(key)

Tenure Months
Monthly Charges
Total Charges
Gender_Female
Gender_Male
Senior Citizen_No
Senior Citizen_Yes
Partner_No
Partner_Yes
Dependents_No
Dependents_Yes
Phone Service_No
Phone Service_Yes
Multiple Lines_No
Multiple Lines_Yes
Internet Service_DSL
Internet Service_Fiber optic
Internet Service_No
Online Security_No
Online Security_Yes
Online Backup_No
Online Backup_Yes
Device Protection_No
Device Protection_Yes
Tech Support_No
Tech Support_Yes
Streaming TV_No
Streaming TV_Yes
Streaming Movies_No
Streaming Movies_Yes
Contract_Month-to-month
Contract_One year
Contract_Two year
Paperless Billing_No
Paperless Billing_Yes
Payment Method_Bank transfer (automatic)
Payment Method_Credit card (automatic)
Payment Method_Electronic check
Payment Method_Mailed check


In [46]:
from pydantic import BaseModel

class CustomerData(BaseModel):

    Gender: str
    Senior_Citizen: str
    Partner: str
    Dependents: str

    Tenure_Months: int

    Phone_Service: str
    Multiple_Lines: str

    Internet_Service: str

    Online_Security: str
    Online_Backup: str
    Device_Protection: str
    Tech_Support: str

    Streaming_TV: str
    Streaming_Movies: str

    Contract: str

    Paperless_Billing: str

    Payment_Method: str

    Monthly_Charges: float

    Total_Charges: float


print("Customer Schema Created Successfully!")

Customer Schema Created Successfully!


In [47]:
print(len(feature_translator))

39


In [48]:

def prepare_customer_dataframe(customer: CustomerData):

    customer_df = pd.DataFrame([{

        "Gender": customer.Gender,
        "Senior Citizen": customer.Senior_Citizen,
        "Partner": customer.Partner,
        "Dependents": customer.Dependents,

        "Tenure Months": customer.Tenure_Months,

        "Phone Service": customer.Phone_Service,
        "Multiple Lines": customer.Multiple_Lines,

        "Internet Service": customer.Internet_Service,

        "Online Security": customer.Online_Security,
        "Online Backup": customer.Online_Backup,
        "Device Protection": customer.Device_Protection,
        "Tech Support": customer.Tech_Support,

        "Streaming TV": customer.Streaming_TV,
        "Streaming Movies": customer.Streaming_Movies,

        "Contract": customer.Contract,

        "Paperless Billing": customer.Paperless_Billing,

        "Payment Method": customer.Payment_Method,

        "Monthly Charges": customer.Monthly_Charges,
        "Total Charges": customer.Total_Charges

    }])

    return customer_df


print("prepare_customer_dataframe() created successfully!")

prepare_customer_dataframe() created successfully!


In [49]:

def predict_customer(customer_df):

    processed = preprocessor.transform(customer_df)

    prediction = int(
        xgboost_model.predict(processed)[0]
    )

    probability = float(
        xgboost_model.predict_proba(processed)[0][1]
    )

    return prediction, probability, processed


print("predict_customer() created successfully!")

predict_customer() created successfully!


In [50]:
def calculate_health_score(probability):

    return round((1 - probability) * 100, 2)


print("calculate_health_score() created successfully!")

calculate_health_score() created successfully!


In [51]:
def calculate_risk_level(probability):

    if probability >= 0.80:
        return "Very High"

    elif probability >= 0.60:
        return "High"

    elif probability >= 0.40:
        return "Medium"

    return "Low"


print("calculate_risk_level() created successfully!")

calculate_risk_level() created successfully!


In [52]:
def calculate_health_status(health_score):

    if health_score >= 80:
        return "🟢 Healthy"

    elif health_score >= 60:
        return "🟡 Stable"

    elif health_score >= 40:
        return "🟠 Needs Attention"

    return "🔴 Critical"


print("calculate_health_status() created successfully!")

calculate_health_status() created successfully!


In [53]:
def calculate_revenue(customer):

    return round(
        customer.Monthly_Charges * 12,
        2
    )


print("calculate_revenue() created successfully!")

calculate_revenue() created successfully!


In [54]:
def calculate_priority(health_score, revenue):

    if health_score < 40 and revenue >= 1000:
        return "Critical"

    elif health_score < 60:
        return "High"

    elif health_score < 80:
        return "Medium"

    return "Low"


print("calculate_priority() created successfully!")

calculate_priority() created successfully!


In [55]:
customer = CustomerData(
    Gender="Male",
    Senior_Citizen="No",
    Partner="No",
    Dependents="No",
    Tenure_Months=2,
    Phone_Service="Yes",
    Multiple_Lines="No",
    Internet_Service="DSL",
    Online_Security="Yes",
    Online_Backup="Yes",
    Device_Protection="No",
    Tech_Support="No",
    Streaming_TV="No",
    Streaming_Movies="No",
    Contract="Month-to-month",
    Paperless_Billing="Yes",
    Payment_Method="Mailed check",
    Monthly_Charges=53.85,
    Total_Charges=108.15
)

customer_df = prepare_customer_dataframe(customer)

prediction, probability, processed = predict_customer(customer_df)

health_score = calculate_health_score(probability)

risk = calculate_risk_level(probability)

status = calculate_health_status(health_score)

revenue = calculate_revenue(customer)

priority = calculate_priority(health_score, revenue)

print("Prediction:", prediction)
print("Probability:", round(probability * 100, 2))
print("Health Score:", health_score)
print("Risk:", risk)
print("Status:", status)
print("Revenue:", revenue)
print("Priority:", priority)

Prediction: 0
Probability: 41.96
Health Score: 58.04
Risk: Medium
Status: 🟠 Needs Attention
Revenue: 646.2
Priority: High


In [56]:
def get_top_drivers(processed_data, top_n=5):

    # Calculate SHAP values
    shap_values = explainer.shap_values(processed_data)

    # Convert to Series
    shap_series = pd.Series(
        np.abs(shap_values[0]),
        index=feature_names
    )

    # Select top features
    top_features = (
        shap_series
        .sort_values(ascending=False)
        .head(top_n)
    )

    drivers = []

    for feature, impact in top_features.items():

        if impact >= 0.50:
            level = "Very High"

        elif impact >= 0.30:
            level = "High"

        elif impact >= 0.15:
            level = "Medium"

        else:
            level = "Low"

        drivers.append({

            "feature":
                feature_translator.get(feature, feature),

            "impact_score":
                round(float(impact), 3),

            "impact_level":
                level
        })

    return drivers


print("get_top_drivers() created successfully!")

get_top_drivers() created successfully!


In [57]:
drivers = get_top_drivers(processed)

drivers

[{'feature': 'Customer is on a month-to-month contract.',
  'impact_score': 0.556,
  'impact_level': 'Very High'},
 {'feature': 'Customer has relatively short tenure.',
  'impact_score': 0.528,
  'impact_level': 'Very High'},
 {'feature': 'Customer has not subscribed to Online Security.',
  'impact_score': 0.357,
  'impact_level': 'High'},
 {'feature': 'Customer uses Fiber Optic internet.',
  'impact_score': 0.235,
  'impact_level': 'Medium'},
 {'feature': 'Customer has high lifetime value.',
  'impact_score': 0.218,
  'impact_level': 'Medium'}]

In [58]:
recommendation_engine = {

    "Customer is on a month-to-month contract.": {
        "department": "Retention Team",
        "priority": "High",
        "action": "Offer discounted annual contract."
    },

    "Customer has relatively short tenure.": {
        "department": "Retention Team",
        "priority": "High",
        "action": "Assign onboarding specialist."
    },

    "Customer has not subscribed to Online Security.": {
        "department": "Sales",
        "priority": "High",
        "action": "Offer complimentary Online Security trial."
    },

    "Customer uses Fiber Optic internet.": {
        "department": "Technical Support",
        "priority": "Medium",
        "action": "Monitor service quality and proactively engage."
    },

    "Customer has high lifetime value.": {
        "department": "Retention Team",
        "priority": "Medium",
        "action": "Prioritize personalized retention campaign."
    }

}

print("Recommendation Engine Loaded!")

Recommendation Engine Loaded!


In [59]:
def generate_recommendations(top_drivers):

    recommendations = []

    for driver in top_drivers:

        feature = driver["feature"]

        if feature in recommendation_engine:

            recommendation = recommendation_engine[feature]

            recommendations.append({

                "feature": feature,

                "department":
                    recommendation["department"],

                "priority":
                    recommendation["priority"],

                "action":
                    recommendation["action"]

            })

    return recommendations


print("generate_recommendations() created successfully!")

generate_recommendations() created successfully!


In [60]:
recommendations = generate_recommendations(drivers)

recommendations

[{'feature': 'Customer is on a month-to-month contract.',
  'department': 'Retention Team',
  'priority': 'High',
  'action': 'Offer discounted annual contract.'},
 {'feature': 'Customer has relatively short tenure.',
  'department': 'Retention Team',
  'priority': 'High',
  'action': 'Assign onboarding specialist.'},
 {'feature': 'Customer has not subscribed to Online Security.',
  'department': 'Sales',
  'priority': 'High',
  'action': 'Offer complimentary Online Security trial.'},
 {'feature': 'Customer uses Fiber Optic internet.',
  'department': 'Technical Support',
  'priority': 'Medium',
  'action': 'Monitor service quality and proactively engage.'},
 {'feature': 'Customer has high lifetime value.',
  'department': 'Retention Team',
  'priority': 'Medium',
  'action': 'Prioritize personalized retention campaign.'}]

In [61]:
def generate_executive_summary(
    prediction,
    probability,
    health_score,
    health_status,
    risk_level,
    top_drivers
):

    prediction_text = "Churn" if prediction == 1 else "Stay"

    summary = f"""
Customer Health Summary

The customer is predicted to {prediction_text}
with a churn probability of {round(probability*100,2)}%.

Health Score: {health_score}/100
Health Status: {health_status}
Risk Level: {risk_level}

Key Drivers:
"""

    for driver in top_drivers:

        summary += f"\n• {driver['feature']}"

    summary += """

Recommended Strategy

• Contact the customer proactively.
• Execute the recommended retention actions.
• Monitor customer engagement during the next billing cycle.
"""

    return summary


print("Executive Summary Generator created successfully!")

Executive Summary Generator created successfully!


In [62]:
summary = generate_executive_summary(
    prediction,
    probability,
    health_score,
    status,
    risk,
    drivers
)

print(summary)


Customer Health Summary

The customer is predicted to Stay
with a churn probability of 41.96%.

Health Score: 58.04/100
Health Status: 🟠 Needs Attention
Risk Level: Medium

Key Drivers:

• Customer is on a month-to-month contract.
• Customer has relatively short tenure.
• Customer has not subscribed to Online Security.
• Customer uses Fiber Optic internet.
• Customer has high lifetime value.

Recommended Strategy

• Contact the customer proactively.
• Execute the recommended retention actions.
• Monitor customer engagement during the next billing cycle.



In [63]:
def generate_api_report(customer):

    # -----------------------------
    # Prepare Customer
    # -----------------------------

    customer_df = prepare_customer_dataframe(customer)

    prediction, probability, processed = predict_customer(customer_df)

    health_score = calculate_health_score(probability)

    risk = calculate_risk_level(probability)

    status = calculate_health_status(health_score)

    revenue = calculate_revenue(customer)

    priority = calculate_priority(
        health_score,
        revenue
    )

    # -----------------------------
    # AI Explainability
    # -----------------------------

    drivers = get_top_drivers(processed)

    recommendations = generate_recommendations(drivers)

    executive_summary = generate_executive_summary(
        prediction,
        probability,
        health_score,
        status,
        risk,
        drivers
    )

    # -----------------------------
    # Final Report
    # -----------------------------

    return {

        "customer": {

            "prediction":
            "Churn" if prediction else "Stay",

            "churn_probability":
            round(probability * 100, 2),

            "health_score":
            health_score,

            "health_status":
            status,

            "risk_level":
            risk,

            "customer_priority":
            priority

        },

        "financials": {

            "annual_revenue_at_risk":
            revenue

        },

        "insights": {

            "top_drivers":
            drivers,

            "recommendations":
            recommendations,

            "executive_summary":
            executive_summary

        }

    }


print("Enterprise API Report Generator created successfully!")

Enterprise API Report Generator created successfully!


In [64]:
report = generate_api_report(customer)

report

{'customer': {'prediction': 'Stay',
  'churn_probability': 41.96,
  'health_score': 58.04,
  'health_status': '🟠 Needs Attention',
  'risk_level': 'Medium',
  'customer_priority': 'High'},
 'financials': {'annual_revenue_at_risk': 646.2},
 'insights': {'top_drivers': [{'feature': 'Customer is on a month-to-month contract.',
    'impact_score': 0.556,
    'impact_level': 'Very High'},
   {'feature': 'Customer has relatively short tenure.',
    'impact_score': 0.528,
    'impact_level': 'Very High'},
   {'feature': 'Customer has not subscribed to Online Security.',
    'impact_score': 0.357,
    'impact_level': 'High'},
   {'feature': 'Customer uses Fiber Optic internet.',
    'impact_score': 0.235,
    'impact_level': 'Medium'},
   {'feature': 'Customer has high lifetime value.',
    'impact_score': 0.218,
    'impact_level': 'Medium'}],
  'recommendations': [{'feature': 'Customer is on a month-to-month contract.',
    'department': 'Retention Team',
    'priority': 'High',
    'action'

In [65]:
from fastapi import FastAPI

app = FastAPI(

    title="Enterprise AI Relationship Manager",

    description="""
Enterprise Customer Success Intelligence Platform

Features:
- Churn Prediction
- Customer Health Score
- SHAP Explainability
- AI Recommendations
- Executive Summary
- Revenue at Risk
""",

    version="2.0"

)

print("FastAPI App Created Successfully!")

FastAPI App Created Successfully!


In [66]:
@app.get("/")

def home():

    return {

        "application":
        "Enterprise AI Relationship Manager",

        "version":
        "2.0",

        "status":
        "Running"

    }

print("Home endpoint created!")

Home endpoint created!


In [67]:
@app.get("/health")

def health():

    return {

        "status":"healthy",

        "model_loaded":True,

        "preprocessor_loaded":True,

        "shap_loaded":True

    }

print("Health endpoint created!")

Health endpoint created!


In [68]:
@app.post("/predict")

def predict(customer: CustomerData):

    customer_df = prepare_customer_dataframe(customer)

    prediction, probability, _ = predict_customer(customer_df)

    return {

        "prediction":
        "Churn" if prediction else "Stay",

        "churn_probability":
        round(probability*100,2)

    }

print("Prediction endpoint created!")

Prediction endpoint created!


In [69]:
@app.post("/customer-report")

def customer_report(customer: CustomerData):

    return generate_api_report(customer)

print("Customer Report endpoint created!")

Customer Report endpoint created!


In [70]:
from pyngrok import ngrok

ngrok.set_auth_token("3HZGabZyMndMOF3tSRb30gn892w_7tyZ2VR8ZvBqxuZcyhcsy")

print("ngrok authenticated!")

ngrok authenticated!


In [71]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

public_url = ngrok.connect(8000)

print("Public URL:", public_url)

def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

thread = threading.Thread(target=run)
thread.start()

Public URL: NgrokTunnel: "https://remold-swinging-gray.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [9452]
INFO:     Waiting for application startup.
